In [94]:
from kilosort_pipeline.utils import load_config, setup, parse_openephys_folders
from kilosort_pipeline.sync import match_chirp_edges
import re
import spikeinterface.extractors as se

from pathlib import Path
from collections import defaultdict
from loguru import logger
import numpy as np
import pynapple as nap
from scipy.interpolate import make_interp_spline

def log_ts(timestamps, name):
    start_ts = timestamps[0]
    end_ts = timestamps[-1]
    logger.info(f"{name}: {start_ts:.4f} ... {end_ts:.4f} s ({timestamps.size} samples)")

def get_kilosort_spikes(output_path, probe_filter=None):
    spike_times_dict = {}

    kilosort_files = list(output_path.glob('*/kilosort/spike_times.npy'))
    if not kilosort_files:
        logger.error("No Kilosort output found. Run Kilosort first.")
        raise FileNotFoundError(f"No spike_times.npy files in {output_path}")
    
    for spike_file in kilosort_files:
        probe_name = spike_file.parent.parent.name

        # Filter probes if requested
        if probe_filter and probe_name not in probe_filter:
            continue

        spike_times_dict[probe_name] = np.load(spike_file, mmap_mode='r')
        logger.info(f"Loaded {len(spike_times_dict[probe_name])} spikes from {probe_name}")

    return spike_times_dict

In [99]:
paths = ['/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual',
  '/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_13-41-27_4Probe_RSC_ADn_RecOpenField',
  '/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_14-22-26_4Probe_RSC_ADn_RecOpenField',
  '/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_14-57-57_4Probe_RSC_ADn_RecOpenField']

probe_filter = ['ProbeA', 'ProbeB']

In [ ]:
ps = parse_openephys_folders(paths, probe_filter)

2025-11-01 16:12:18 | INFO     | Parsing OpenEphys folders
2025-11-01 16:13:45 | SUCCESS  | Parsed 3 stream(s) across 4 session(s)


In [186]:
spike_times = get_kilosort_spikes(p['local_output'], probe_filter=p['probe_filter'])
spike_times

2025-10-31 19:51:54 | INFO     | Loaded 110944227 spikes from ProbeA
2025-10-31 19:51:54 | INFO     | Loaded 164597833 spikes from ProbeB


{'ProbeA': memmap([         4,         14,         14, ..., 1010766696, 1010766697,
         1010766703], shape=(110944227,)),
 'ProbeB': memmap([        11,         13,         17, ..., 1027373483, 1027373486,
         1027373494], shape=(164597833,))}

In [227]:
class Timestamps:
    def __init__(self, fs=30000.0):
        self.fs = fs
        self.dt = 1 / fs
        self.timestamps = []
        self.cont_ranges = []  # Store continuous timestamp ranges in global time
        self.t_last = 0.0
        self.starting_states = []
        self.ranges = []

    def update(self, event_ts, cont_ts, starting_state, global_start=None):
        """
        Update with a new segment.
        
        Parameters:
        -----------
        event_ts : array
            Event timestamps in local time
        cont_ts : array
            Continuous timestamps in local time
        starting_state : int
            Starting state of this segment
        global_start : float, optional
            Explicit global start time. If None, uses t_last + dt
        """
        # Determine global continuous start
        if global_start is not None:
            global_cont_start = global_start
        else:
            global_cont_start = self.t_last + self.dt
        
        # Calculate event timestamps in global time
        global_event_ts = global_cont_start + event_ts - cont_ts[0]
        self.timestamps.append(global_event_ts)
        
        # Calculate continuous range in global time
        global_cont_end = global_cont_start + cont_ts[-1] - cont_ts[0]
        self.cont_ranges.append((global_cont_start, global_cont_end))
        
        # Update t_last for next segment
        self.t_last = global_cont_end
        
        self.starting_states.append(starting_state)
        self.ranges.append((cont_ts[0], cont_ts[-1]))
    
    def get_global_timestamps(self):
        return np.concatenate(self.timestamps)
    
    def get_segment(self, segment_id):
        return self.timestamps[segment_id]
    
    def get_cont_range(self, segment_id):
        """Get the global continuous timestamp range for a segment"""
        return self.cont_ranges[segment_id]
    
    def __repr__(self):
        return f"Timestamps(Segments: {len(self.timestamps)}, fs={self.fs})"

def load_events(events_path, cont_path):
    event_ts  = np.load(events_path, mmap_mode='r')
    cont_ts   = np.load(cont_path, mmap_mode='r')
    states    = np.load(events_path.replace('timestamps.npy', 'states.npy'), mmap_mode='r')
    return event_ts, cont_ts, states


In [228]:
ADC = Timestamps(fs=30300.0)

adc_event_paths = ps["timestamps"]["OneBox-ADC"]['event']
adc_cont_paths = ps["timestamps"]["OneBox-ADC"]['cont']

for idx, (event_path, cont_path) in enumerate(zip(adc_event_paths, adc_cont_paths)):
    event_ts, cont_ts, states = load_events(event_path, cont_path)
    
    # First segment starts at 0.0, others continue from previous
    if idx == 0:
        ADC.update(event_ts, cont_ts, int(states[0]), global_start=0.0)
    else:
        ADC.update(event_ts, cont_ts, int(states[0]))
    
    ########## LOGGING #################################
    log_ts(event_ts, "ADC event")
    log_ts(cont_ts, "ADC cont")
    logger.info(f"Starting state: {ADC.starting_states[-1]}")
    log_ts(ADC.get_segment(-1), "ADC global segment")
    log_ts(ADC.get_global_timestamps(), "ADC global")
    logger.info("-"*60)


2025-11-01 00:28:01 | INFO     | ADC event: 12.0830 ... 3645.1092 s (7267 samples)
2025-11-01 00:28:01 | INFO     | ADC cont: 11.6334 ... 3645.4310 s (110114290 samples)
2025-11-01 00:28:01 | INFO     | Starting state: -1
2025-11-01 00:28:01 | INFO     | ADC global segment: 0.4497 ... 3633.4758 s (7267 samples)
2025-11-01 00:28:01 | INFO     | ADC global: 0.4497 ... 3633.4758 s (7267 samples)
2025-11-01 00:28:01 | INFO     | ------------------------------------------------------------
2025-11-01 00:28:01 | INFO     | ADC cont: 11.6334 ... 3645.4310 s (110114290 samples)
2025-11-01 00:28:01 | INFO     | Starting state: -1
2025-11-01 00:28:01 | INFO     | ADC global segment: 0.4497 ... 3633.4758 s (7267 samples)
2025-11-01 00:28:01 | INFO     | ADC global: 0.4497 ... 3633.4758 s (7267 samples)
2025-11-01 00:28:01 | INFO     | ------------------------------------------------------------
2025-11-01 00:28:01 | INFO     | ADC event: 123.0811 ... 2266.3990 s (11788 samples)
2025-11-01 00:28:0

In [229]:
probe_filter = ['ProbeA', 'ProbeB']
probe_timestamps = {k:d for k,d in ps["timestamps"].items() if k != "OneBox-ADC" and k in probe_filter}
synced_spikes = []

masks = {}
for probe, paths in probe_timestamps.items():
    PRB = Timestamps(fs=30000.0)
    logger.info(f"Processing probe: {probe}")

    logger.info("Extracting kilosort spikes")
    kilosort_spikes = spike_times[probe] / PRB.fs
    log_ts(kilosort_spikes, f"{probe} spikes")
    total_spikes_left = kilosort_spikes.size
    logger.info('='*60)

    masks[probe] = []
    for idx, (ev_path, cont_path) in enumerate(zip(paths['event'], paths['cont'])):
        pr_event_ts, pr_cont_ts, pr_states = load_events(ev_path, cont_path)

        # Handle state mismatches
        if pr_states[0] != ADC.starting_states[idx]:
            logger.warning(f"State mismatch between {probe} and ADC")
            logger.info("Matching edges")
            pr_event_ts, _ = match_chirp_edges(pr_event_ts, ADC.get_segment(idx))

        # First segment starts at 0.0, others continue from previous
        if idx == 0:
            PRB.update(pr_event_ts, pr_cont_ts, int(pr_states[0]), global_start=0.0)
        else:
            PRB.update(pr_event_ts, pr_cont_ts, int(pr_states[0]))

        ########## LOGGING #################################
        log_ts(pr_event_ts, f"Event timestamps")
        log_ts(pr_cont_ts, f"Continuous timestamps")
        logger.info(f"Starting state: {int(pr_states[0])}")
        log_ts(PRB.get_segment(-1), f"Global segment")
        log_ts(PRB.get_global_timestamps(), f"Global")
        ########## LOGGING #################################

        probe_times = PRB.get_segment(idx)
        adc_times = ADC.get_segment(idx)
        
        # Get continuous timestamp ranges for spike extraction
        probe_cont_start, probe_cont_end = PRB.get_cont_range(idx)
        logger.info(f"Continuous range: {probe_cont_start:.4f} - {probe_cont_end:.4f} s")

        # Extract spikes for this segment using continuous range
        if idx == 0:
            # First segment: start from beginning of continuous data
            mask = (kilosort_spikes >= probe_cont_start) & (kilosort_spikes <= probe_cont_end)
        else:
            # Subsequent segments: exclude start boundary to avoid overlap
            mask = (kilosort_spikes > probe_cont_start) & (kilosort_spikes <= probe_cont_end)
        
        masks[probe].append(mask)
        probe_spikes = kilosort_spikes[mask]
        log_ts(probe_spikes, f"Extracted spikes")
        
        # Handle length mismatches
        if len(probe_times) < len(adc_times):
            logger.warning(f"  Truncating ADC timestamps. ADC timestamps: {len(adc_times)} -> {len(probe_times)}.")
            adc_times = adc_times[:len(probe_times)]
        elif len(adc_times) < len(probe_times):
            logger.warning(f"  ADC timestamps shorter than probe timestamps. Truncating probe timestamps.")
            probe_times = probe_times[:len(adc_times)]
        
        # Interpolate/extrapolate to ADC time
        spl = make_interp_spline(x=probe_times, y=adc_times, k=1)
        adc_spikes = spl(probe_spikes)
        synced_spikes.append(adc_spikes)
        total_spikes_left -= adc_spikes.size

        log_ts(adc_spikes, "ADC interpolated spikes")
        logger.info(f"Synced spikes: {adc_spikes.size}/{kilosort_spikes.size}. Remaining spikes: {total_spikes_left}")
        logger.info("="*60)


2025-11-01 00:28:06 | INFO     | Processing probe: ProbeA
2025-11-01 00:28:06 | INFO     | Extracting kilosort spikes
2025-11-01 00:28:06 | INFO     | Extracting kilosort spikes
2025-11-01 00:28:06 | INFO     | ProbeA spikes: 0.0001 ... 33692.2234 s (110944227 samples)
2025-11-01 00:28:06 | INFO     | ============================================================
2025-11-01 00:28:06 | INFO     | ProbeA spikes: 0.0001 ... 33692.2234 s (110944227 samples)
2025-11-01 00:28:06 | INFO     | ============================================================
2025-11-01 00:28:06 | INFO     | Event timestamps: 12.0831 ... 3645.1092 s (7267 samples)
2025-11-01 00:28:06 | INFO     | Continuous timestamps: 11.6301 ... 3645.4102 s (109013404 samples)
2025-11-01 00:28:06 | INFO     | Starting state: -1
2025-11-01 00:28:06 | INFO     | Global segment: 0.4530 ... 3633.4791 s (7267 samples)
2025-11-01 00:28:06 | INFO     | Global: 0.4530 ... 3633.4791 s (7267 samples)
2025-11-01 00:28:06 | INFO     | Continuou

In [ ]:
# Diagnostic: Check where the missing spikes are
for probe in probe_filter:
    kilosort_spikes = spike_times[probe] / 30000.0
    total_spikes = kilosort_spikes.size
    
    # Count spikes assigned to all segments
    total_assigned = sum(np.sum(mask) for mask in masks[probe])
    missing = total_spikes - total_assigned
    
    logger.info(f"\n{'='*60}")
    logger.info(f"{probe} Missing Spikes Analysis:")
    logger.info(f"  Total Kilosort spikes: {total_spikes:,}")
    logger.info(f"  Spikes assigned to segments: {total_assigned:,}")
    logger.info(f"  Missing spikes: {missing:,}")
    logger.info(f"  Kilosort time range: {kilosort_spikes[0]:.4f} - {kilosort_spikes[-1]:.4f} s")
    
    # Check where missing spikes are
    spikes_before_zero = np.sum(kilosort_spikes < 0.0)
    logger.info(f"  Spikes before t=0.0s: {spikes_before_zero:,}")
    
    # Check for gaps between segments
    for i, mask in enumerate(masks[probe]):
        segment_spikes = kilosort_spikes[mask]
        if segment_spikes.size > 0:
            logger.info(f"  Segment {i}: {segment_spikes[0]:.4f} - {segment_spikes[-1]:.4f} s ({segment_spikes.size:,} spikes)")
    
    logger.info(f"{'='*60}\n")


In [212]:
PRB.get_segment(idx)

array([ 7736.38360023,  7736.7471184 ,  7737.09080843, ...,
       34248.09779629, 34248.24310201, 34248.3853475 ], shape=(145696,))

In [213]:
PRB.get_segment(idx-1)

array([5777.23395149, 5777.38591485, 5777.53629438, ..., 7735.71027625,
       7735.8817911 , 7736.05120561], shape=(10765,))

In [214]:
sum = 0
for mask in masks['ProbeA']:
    sum += np.sum(mask)
sum

np.int64(110937983)

In [215]:
spike_times['ProbeA']

memmap([         4,         14,         14, ..., 1010766696, 1010766697,
        1010766703], shape=(110944227,))

In [131]:
from kilosort_pipeline.sync import match_chirp_edges

probe_timestamps = {k: d for k, d in timestamps.items() if k != "OneBox-ADC"}
logger.info("COMPUTING GLOBAL TIMESTAMPS")

probe_global_timestamps = {}
for probe_name, paths in probe_timestamps.items():
    event_ts, cont_ts = paths['event'], paths['cont']
    probe_ts = Timestamps(fs=30000.0)
    logger.info(f"Processing probe: {probe_name}")

    for idx, (ev_path, ct_path) in enumerate(zip(event_ts, cont_ts)):
        # Load states
        ev_states = np.load(ev_path.replace('timestamps.npy', 'states.npy'), mmap_mode='r')
        ct_states = np.load(ct_path.replace('timestamps.npy', 'states.npy'), mmap_mode='r')

        # Load timestamps
        ev_ts = np.load(ev_path, mmap_mode='r')
        ct_ts = np.load(ct_path, mmap_mode='r')

        log_ts(ev_ts, f"    event")
        log_ts(ct_ts, f"    cont")

        if ev_states[0] != ct_states[0]:
            logger.warning(f"   Mismatch in initial states.")
            ev_ts = match_chirp_edges(ev_ts, ADC.get_segment(idx), ct_ts)
        else:
            logger.info(f"  Initial states match.")
        
        probe_ts.update(ev_ts, ct_ts)
        log_ts(probe_ts.get_segment(-1), f"    {probe_name} global segment")
        log_ts(probe_ts.get_total(), f"    {probe_name} global")
        logger.info("-"*60)

2025-10-31 17:36:57 | INFO     | COMPUTING GLOBAL TIMESTAMPS
2025-10-31 17:36:57 | INFO     | Processing probe: ProbeA


FileNotFoundError: [Errno 2] No such file or directory: '\\\\fsmresfiles.fsm.northwestern.edu\\FSMResfiles\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\VisualStimuli\\AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual\\Record Node 103\\experiment1\\recording1\\continuous\\OneBox-106.ProbeA\\states.npy'

In [128]:
probe_timestamps

{'ProbeA': {'event': ['\\\\fsmresfiles.fsm.northwestern.edu\\FSMResfiles\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\VisualStimuli\\AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual\\Record Node 103\\experiment1\\recording1\\events\\OneBox-106.ProbeA\\TTL\\timestamps.npy',
   '\\\\fsmresfiles.fsm.northwestern.edu\\FSMResfiles\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\OpenField_Homecage\\AA001_2025-10-09_13-41-27_4Probe_RSC_ADn_RecOpenField\\Record Node 117\\experiment1\\recording1\\events\\OneBox-121.ProbeA\\TTL\\timestamps.npy',
   '\\\\fsmresfiles.fsm.northwestern.edu\\FSMResfiles\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\OpenField_Homecage\\AA001_2025-10-09_14-22-26_4Probe_RSC_ADn_RecOpenField\\Record Node 117\\experiment1\\recording1\\events\\OneBox-121.ProbeA\\TTL\\timestamps.npy',
   '\\\\fsmresfiles.fsm.northwestern.edu\\FSMResfiles\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\OpenField_Homecage\\AA001_2025-10-09_14-57-57_4Probe_RSC_ADn_RecOpenFi

In [30]:
from kilosort_pipeline.sync import match_chirp_edges

timestamps = ps['timestamps']
logger.info("COMPUTING GLOBAL TIMESTAMPS")

# Get ADC timestamps for reference
adc_name = 'OneBox-ADC'
if adc_name not in timestamps:
    logger.error(f"ADC stream '{adc_name}' not found in timestamps")
    raise ValueError(f"ADC stream '{adc_name}' not found")

adc_event_paths = timestamps[adc_name]['event']
adc_cont_paths = timestamps[adc_name]['cont']
global_events = defaultdict(list)
dt = 1 / fs

num_recordings = len(adc_event_paths)
for rec_idx in range(num_recordings):
    logger.info(f"Recording {rec_idx + 1}/{num_recordings}:")

    # Load ADC timestamps
    adc_event = np.load(adc_event_paths[rec_idx], mmap_mode='r')
    adc_cont = np.load(adc_cont_paths[rec_idx], mmap_mode='r')

    for probe in sorted(timestamps.keys()):
        logger.info(f"  Probe: {probe}")

        # Load probe timestamps
        event = np.load(timestamps[probe]['event'][rec_idx], mmap_mode='r')
        cont = np.load(timestamps[probe]['cont'][rec_idx], mmap_mode='r')

        if "ADC" not in probe:
            # Check first edges
            event_state_path = timestamps[probe]['event'][rec_idx].replace('timestamps.npy', 'states.npy')
            adc_state_path = adc_event_paths[rec_idx].replace('timestamps.npy', 'states.npy')

            event_state = np.load(event_state_path, mmap_mode='r')
            adc_state = np.load(adc_state_path, mmap_mode='r')

            if event_state[0] != adc_state[0]:
                logger.info(f"    Initial states differ (probe={event_state[0]}, ADC={adc_state[0]})")
                logger.info("    Aligning to ADC reference...")

                # Match ADC and probe timestamps
                event, _ = match_chirp_edges(event, adc_event)
            else:
                logger.info(f"    Initial states match (state={event_state[0], adc_state[0]})")

        # Compute global timestamps
        global_ts = event - cont[0] + last_ts[probe]

        logger.info(f"    Event range: {event[0]} ... {event[-1]}")
        logger.info(f"    Continuous range: {cont[0]} ... {cont[-1]}")
        logger.info(f"    Global timestamps: {global_ts[0]:.2f} ... {global_ts[-1]:.2f} s")

        # Update last_ts for next recording
        last_ts[probe] += cont[-1] - cont[0] + dt

        # Store global events
        global_events[probe].append(global_ts)


2025-10-31 15:15:59 | INFO     | Computing global timestamps
2025-10-31 15:15:59 | INFO     | Computing global timestamps
2025-10-31 15:15:59 | INFO     | Recording 1/4:
2025-10-31 15:15:59 | INFO     |   Probe: OneBox-ADC
2025-10-31 15:15:59 | INFO     |     Event range: 12.083016833316837 ... 3645.10918514046
2025-10-31 15:15:59 | INFO     |     Continuous range: 11.633358383658386 ... 3645.4310364512826
2025-10-31 15:15:59 | INFO     |     Global timestamps: 0.45 ... 3633.48 s
2025-10-31 15:15:59 | INFO     |   Probe: ProbeA
2025-10-31 15:15:59 | INFO     |     Initial states match (state=-1)
2025-10-31 15:15:59 | INFO     |     Event range: 12.0831 ... 3645.1091666666666
2025-10-31 15:15:59 | INFO     |     Continuous range: 11.6301 ... 3645.410199999938
2025-10-31 15:15:59 | INFO     |     Global timestamps: 0.45 ... 3633.48 s
2025-10-31 15:15:59 | INFO     |   Probe: ProbeB
2025-10-31 15:15:59 | INFO     |     Initial states match (state=-1)
2025-10-31 15:15:59 | INFO     |     E

In [55]:
fs = p['fs']

# Extract ADC reference timestamps
if 'OneBox-ADC' in global_events:
    adc_events = global_events.pop('OneBox-ADC')
synced_spikes = {}

for probe_name, spike_times in spike_times_dict.items():
    logger.info(f"Processing {probe_name}")
    
    if probe_name not in global_events:
        logger.warning(f"  No global timestamps found for {probe_name}. Skipping.")
        continue
    
    # Convert spike times to seconds
    spike_times_sec = nap.Ts(t=spike_times / fs, time_units='s')
    log_ts(spike_times_sec.t, name='Kilosort Spikes')
    adc_spikes_total = []
    
    probe_spikes_size = spike_times_sec.t.size
    adc_spikes_size = 0
    probe_events = global_events[probe_name]
    for session_idx, (probe_times, adc_times) in enumerate(zip(probe_events, adc_events), 1):
        logger.info(f"Session {session_idx}/{len(probe_events)}")
        # Define overlapping epochs
        probe_len = probe_times.size
        adc_len = adc_times.size
        
        log_ts(probe_times, 'Probe Global Timestamps')
        log_ts(adc_times, 'ADC Global Timestamps')

        if probe_len < adc_len:
            adc_times = adc_times[:probe_len]
        elif adc_len < probe_len:
            logger.warning(f"  ADC timestamps shorter than probe timestamps. Truncating probe timestamps.")
            probe_times = probe_times[:adc_len]

        # Restrict spikes to overlap
        pr_spikes = spike_times_sec.restrict(nap.IntervalSet(probe_times[0], probe_times[-1]))
        log_ts(pr_spikes.t, 'Original Spikes (Probe)')
        logger.info(f"  Remaining spikes: {probe_spikes_size - pr_spikes.t.size}")
        
        # Linear interpolation to ADC time base
        spl = make_interp_spline(probe_times, adc_times, k=1)
        adc_spikes = spl(pr_spikes.t)
        adc_spikes_total.append(adc_spikes)
        log_ts(adc_spikes, 'Interpolated Spikes (ADC)')

        adc_spikes_size += adc_spikes.size
        probe_spikes_size -= adc_spikes.size
    
    # Concatenate all sessions
    synced_spikes[probe_name] = np.concatenate(adc_spikes_total)

    logger.success(f"  Synced {adc_spikes_size} spikes for {probe_name} ({spike_times.size} total spikes)")
    log_ts(synced_spikes[probe_name], 'Interpolated Spikes')

    # Save if output path provided
    # if output_path:
    #     sync_file = output_path / probe_name / "spike_times_synced.npy"
    #     np.save(sync_file, synced_spikes[probe_name])
    #     logger.info(f"  Saved to: {sync_file}")

2025-10-31 16:20:44 | INFO     | Processing ProbeA
2025-10-31 16:20:45 | INFO     |   Kilosort Spikes range: 0.000133333 ... 33692.223433333 s (110944227 samples)
2025-10-31 16:20:46 | INFO     | Session 1/4
2025-10-31 16:20:46 | INFO     |   Probe Global Timestamps range: 0.4529999999999994 ... 3633.4790666666668 s (7267 samples)
2025-10-31 16:20:46 | INFO     |   ADC Global Timestamps range: 0.44965844965845037 ... 3633.4758267568013 s (7267 samples)
2025-10-31 16:20:46 | INFO     |   Original Spikes (Probe) range: 0.4533 ... 3633.478933333 s (9685902 samples)
2025-10-31 16:20:46 | INFO     |   Remaining spikes: 101258325
2025-10-31 16:20:46 | INFO     |   Interpolated Spikes (ADC) range: 0.44995845955846087 ... 3633.4756934179104 s (9685902 samples)
2025-10-31 16:20:46 | INFO     | Session 2/4
2025-10-31 16:20:46 | INFO     |   Probe Global Timestamps range: 3633.827304173452 ... 5223.316814867998 s (8742 samples)
2025-10-31 16:20:46 | INFO     |   ADC Global Timestamps range: 3633.

In [63]:
spike_times_sec.get(probe_times[0], probe_times[-1])

Time (s)
7736.383966667
7736.384066667
7736.3842
7736.384233333
7736.384233333
7736.3845
7736.384533333
...
34245.782366667
34245.782566667
34245.782633333
34245.782733333
34245.782766667
34245.782866667
34245.783133333
shape: 122519429

In [64]:
spike_times_sec.restrict(nap.IntervalSet(probe_times[0], probe_times[-1]))

Time (s)
7736.383966667
7736.384066667
7736.3842
7736.384233333
7736.384233333
7736.3845
7736.384533333
...
34245.782366667
34245.782566667
34245.782633333
34245.782733333
34245.782766667
34245.782866667
34245.783133333
shape: 122519429

In [65]:
np.where((spike_times_sec.t >= probe_times[0]) & (spike_times_sec.t <= probe_times[-1]))

(array([ 42078404,  42078405,  42078406, ..., 164597830, 164597831,
        164597832], shape=(122519429,)),)

In [76]:
start = np.round(probe_times[0] * fs).astype(np.int32)
end = np.round(probe_times[-1] * fs).astype(np.int32)
np.where((spike_times >= start) & (spike_times <= end))[0]

array([ 42078404,  42078405,  42078406, ..., 164597830, 164597831,
       164597832], shape=(122519429,))